# Run Numerical Experiments
This is the entry point for the formal experiments. Before running, verify the configuration and input data. Execution will write files with the same names to the data/ and results/ directories. The reproducible experiments do not require network access.

In [1]:
import json, sys, types
from pathlib import Path

def _load_core_notebook():
    notebook_path = Path("00_bn_core.ipynb")
    document = json.loads(notebook_path.read_text(encoding="utf-8"))
    module = types.ModuleType("bn_core")
    module.__file__ = str(notebook_path.resolve())
    sys.modules["bn_core"] = module
    for cell_index, cell in enumerate(document["cells"]):
        if cell.get("cell_type") == "code":
            source = "".join(cell.get("source", []))
            exec(compile(source, f"{notebook_path}#cell-{cell_index}", "exec"), module.__dict__)
    return module

bn_core = _load_core_notebook()


In [2]:
"""Reproduce every new numerical claim in the round-2 manuscript.

Usage: python run_experiments.py
Writes fixed CPTs, saved orders/samples, raw per-repeat timing, per-fit losses,
selection decisions, and summaries. No network access is needed.
"""
from pathlib import Path
import csv
import hashlib
import json
import math
import platform
import sys
import time
import numpy as np
from bn_core import *

ROOT = Path.cwd().resolve()


def dump_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, allow_nan=False), encoding="utf-8")


def write_csv(path, rows):
    if rows:
        with Path(path).open("w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0]))
            w.writeheader(); w.writerows(rows)


def main():
    config = json.loads((ROOT / "config.json").read_text())
    (ROOT / "results").mkdir(exist_ok=True)
    meta = dict(python=sys.version, numpy=np.__version__, platform=platform.platform(),
                processor=platform.processor(), machine=platform.machine(),
                seed=config["seed"], config_sha256=hashlib.sha256((ROOT/"config.json").read_bytes()).hexdigest(),
                timing="perf_counter_ns; warmup then 10 individual calls per order; constructor only",
                data={}, scripts={})
    for f in sorted(ROOT.glob("*.py")):
        meta["scripts"][f.name] = hashlib.sha256(f.read_bytes()).hexdigest()
    all_summary = {}
    for network_index, (name, spec) in enumerate(config["networks"].items()):
        t0 = time.perf_counter()
        print("NETWORK", name, flush=True)
        data_path = ROOT / "data" / (name + ".bif.gz")
        meta["data"][name] = dict(sha256=hashlib.sha256(data_path.read_bytes()).hexdigest(),
                                url=f"https://www.bnlearn.com/bnrepository/{name}/{name}.bif.gz")
        bn = read_bif(data_path)
        p, cards = len(bn.names), bn.cards
        evidence = {bn.names.index(v): bn.states[bn.names.index(v)].index(x) for v, x in spec["evidence"].items()}
        C, R = set(evidence), set(range(p)) - set(evidence)
        query = tuple(bn.names.index(v) for v in spec["query"])
        event = tuple(bn.states[v].index(x) for v, x in zip(query, spec["event"]))
        target = marginal(bn.factors(), query, cards, evidence)
        evidence_prob = float(marginal(bn.factors(), (), cards, evidence, normalize=False))
        dump_json(ROOT/"data"/f"{name}_parameters.json",
                  dict(names=bn.names, states=bn.states, parents=bn.parents,
                       cpts=[t.tolist() for t in bn.cpts],
                       evidence={bn.names[v]: bn.states[v][x] for v,x in evidence.items()},
                       cpt_row_normalization_max_deviation=bn.normalization_max_deviation,
                       original_file_sha256=meta["data"][name]["sha256"]))
        rng = np.random.default_rng(np.random.SeedSequence([config["seed"], network_index, 0]))
        orders, order_rows, timing_rows, models, edge_to_id = [], [], [], {}, {}
        for k in range(config["orders"]):
            beta = topological_order(p, bn.edges, rng)
            HR, H, operations = fgcr(p, bn.edges, C, beta)
            remaining_order = [v for v in beta if v not in C]
            full_order = [v for v in beta if v in C] + remaining_order
            assert HR == ordered_oracle(p, bn.edges, C, remaining_order)
            assert H == ordered_oracle(p, bn.edges, set(), full_order)
            key = tuple(sorted(HR))
            if key not in edge_to_id:
                graph_id = len(edge_to_id)
                edge_to_id[key] = graph_id
                models[graph_id] = dict(edges=HR, parents=graph_parents(R, HR))
            graph_id = edge_to_id[key]
            fgcr(p, bn.edges, C, beta)  # warmup
            times = []
            for rep in range(config["timing_repeats"]):
                tic = time.perf_counter_ns()
                fgcr(p, bn.edges, C, beta)
                elapsed = (time.perf_counter_ns() - tic) / 1e6
                times.append(elapsed)
                timing_rows.append(dict(network=name, order_id=k, repeat=rep, milliseconds=elapsed))
            dimension = sum((cards[v]-1)*math.prod(cards[u] for u in pa) for v,pa in models[graph_id]["parents"].items())
            order_rows.append(dict(network=name, order_id=k, graph_id=graph_id,
                                   dimension=dimension, edges=len(HR), median_ms=float(np.median(times)), **operations))
            orders.append(dict(order_id=k, graph_id=graph_id, beta=[bn.names[v] for v in beta],
                               alpha_R=[bn.names[v] for v in remaining_order]))
        models[-1] = dict(edges={(u,v) for u,v in bn.edges if u in R and v in R},
                          parents=graph_parents(R, {(u,v) for u,v in bn.edges if u in R and v in R}))
        print("Distinct conditional graphs:", len(edge_to_id), flush=True)
        dump_json(ROOT/"data"/f"{name}_orders.json", orders)
        write_csv(ROOT/"results"/f"{name}_orders.csv", order_rows)
        write_csv(ROOT/"results"/f"{name}_timings.csv", timing_rows)
        family_cache = {}
        known_rows = []
        entropy = None
        exact_error = 0.
        for gid, model in models.items():
            model["true"] = {}
            true_factors = []
            for v, pa in model["parents"].items():
                key = (v, pa)
                if key not in family_cache:
                    family_cache[key] = true_family(bn, evidence, v, pa)
                model["true"][v] = family_cache[key]
                true_factors.append(Factor(pa+(v,), model["true"][v]["theta"].reshape(tuple(cards[u] for u in pa)+(cards[v],))))
            q = marginal(true_factors, query, cards)
            if gid >= 0:
                err = float(np.max(np.abs(q-target)))
                exact_error = max(exact_error, err)
                assert err < 1e-10, ("factorization", name, gid, err)
                e = sum(t["entropy"] for t in model["true"].values())
                if entropy is None: entropy = e
                assert abs(e-entropy) < 1e-9, ("entropy", gid, e, entropy)
            known_rows.append(dict(network=name, graph_id=gid,
                                   query_max_abs_error=float(np.max(np.abs(q-target))),
                                   query_tv=float(.5*np.abs(q-target).sum())))
        write_csv(ROOT/"results"/f"{name}_known_cpt.csv", known_rows)
        fit_rows, selection_rows, pilot_rows, proposal_rows = [], [], [], []
        saved_samples = {}
        q_target = float(target[event])
        for replicate in range(config["replicates"]):
            print(name, "replicate", replicate+1, flush=True)
            rng = np.random.default_rng(np.random.SeedSequence([config["seed"], network_index, 1, replicate]))
            nmax = max(config["sample_sizes"])
            nall = config["pilot_size"] + nmax + config["validation_size"]
            X, proposals = sample_conditional(bn, nall, evidence, rng, evidence_prob)
            saved_samples[f"replicate_{replicate:02d}"] = X
            pilot = X[:config["pilot_size"]]
            train = X[config["pilot_size"]:config["pilot_size"]+nmax]
            validation = X[config["pilot_size"]+nmax:]
            proposal_rows.append(dict(network=name, replicate=replicate, retained=nall, proposals=proposals))
            pilots = {gid: pilot_score(fit_tables(pilot,m["parents"],cards,config["iss"]),len(pilot))
                      for gid,m in models.items() if gid >= 0}
            pilot_id = min(pilots,key=lambda g:(pilots[g],g))
            min_dimension_id = min(order_rows,key=lambda r:(r["dimension"],r["order_id"]))["graph_id"]
            random_id = orders[int(rng.integers(len(orders)))]["graph_id"]
            for gid, score in pilots.items():
                pilot_rows.append(dict(network=name, replicate=replicate, graph_id=gid, score=score))
            for n in config["sample_sizes"]:
                per_graph = {}
                for gid, model in models.items():
                    fit = fit_tables(train[:n], model["parents"], cards, config["iss"])
                    fitted_query = marginal(fitted_factors(fit,cards), query, cards)
                    kl = max(0., cross_entropy(fit,model["true"]) - entropy)
                    loss = logloss(fit,validation,cards)
                    result = dict(network=name, replicate=replicate, n=n, graph_id=gid,
                                  query_estimate=float(fitted_query[event]),
                                  query_truth=q_target,
                                  query_error=float(fitted_query[event]-q_target),
                                  query_tv=float(.5*np.abs(fitted_query-target).sum()),
                                  kl=kl, validation_logloss=loss)
                    if gid >= 0:
                        result.update(weighted_bounds(fit,model["true"],n,config["delta"]))
                        # Theorem is 95% per fixed graph. Selection requires multiplicity correction.
                        simultaneous = weighted_bounds(fit,model["true"],n,config["delta"]/len(edge_to_id))
                        result["selection_empirical_bound"] = simultaneous["empirical_bound"]
                        result["empirical_violation"] = kl > result["empirical_bound"] + 1e-10
                        result["oracle_violation"] = kl > result["oracle_bound"] + 1e-10
                    else:
                        result.update(oracle_bound=None,empirical_bound=None,uniform_bound=None,
                                      uniform_applicable=False,pi_min=None,theta_min=None,
                                      selection_empirical_bound=None,empirical_violation=None,oracle_violation=None)
                    fit_rows.append(result); per_graph[gid] = result
                valid = {g:r for g,r in per_graph.items() if g>=0}
                holdout_id = min(valid,key=lambda g:(valid[g]["validation_logloss"],g))
                bound_id = min(valid,key=lambda g:(valid[g]["selection_empirical_bound"],g))
                oracle_id = min(valid,key=lambda g:(valid[g]["kl"],g))
                for method,gid in dict(holdout=holdout_id,pilot_heuristic=pilot_id,
                                       minimum_dimension=min_dimension_id,random_order=random_id,
                                       empirical_bound=bound_id,oracle_kl=oracle_id,induced=-1).items():
                    r=per_graph[gid]
                    selection_rows.append(dict(network=name,replicate=replicate,n=n,method=method,graph_id=gid,
                                               kl=r["kl"],query_error=r["query_error"],query_tv=r["query_tv"],
                                               excess_kl=r["kl"]-per_graph[oracle_id]["kl"]))
        np.savez_compressed(ROOT/"data"/f"{name}_conditional_samples.npz", **saved_samples)
        write_csv(ROOT/"results"/f"{name}_raw_fits.csv",fit_rows)
        write_csv(ROOT/"results"/f"{name}_selection.csv",selection_rows)
        write_csv(ROOT/"results"/f"{name}_pilot_scores.csv",pilot_rows)
        write_csv(ROOT/"results"/f"{name}_sampling.csv",proposal_rows)
        # One row per sampled order, n and its 20 paired replications; duplicates retained.
        sensitivity=[]
        for order in order_rows:
            for n in config["sample_sizes"]:
                rows=[r for r in fit_rows if r["n"]==n and r["graph_id"]==order["graph_id"]]
                sensitivity.append(dict(**order,n=n,
                                        query_rmse=math.sqrt(np.mean([r["query_error"]**2 for r in rows])),
                                        mean_kl=float(np.mean([r["kl"] for r in rows])),
                                        mean_query_tv=float(np.mean([r["query_tv"] for r in rows]))))
        write_csv(ROOT/"results"/f"{name}_order_sensitivity.csv",sensitivity)
        selection_summary=[]
        for n in config["sample_sizes"]:
            for method in sorted({r["method"] for r in selection_rows}):
                rows=[r for r in selection_rows if r["n"]==n and r["method"]==method]
                selection_summary.append(dict(network=name,n=n,method=method,
                                              mean_kl=float(np.mean([r["kl"] for r in rows])),
                                              sd_kl=float(np.std([r["kl"] for r in rows],ddof=1)),
                                              query_rmse=math.sqrt(np.mean([r["query_error"]**2 for r in rows])),
                                              mean_excess_kl=float(np.mean([r["excess_kl"] for r in rows]))))
        write_csv(ROOT/"results"/f"{name}_selection_summary.csv",selection_summary)
        bound_summary=[]
        for n in config["sample_sizes"]:
            rows=[r for r in fit_rows if r["n"]==n and r["graph_id"]>=0]
            bound_summary.append(dict(n=n,fits=len(rows),
                                      median_kl=float(np.median([r["kl"] for r in rows])),
                                      median_oracle_bound=float(np.median([r["oracle_bound"] for r in rows])),
                                      median_empirical_bound=float(np.median([r["empirical_bound"] for r in rows])),
                                      uniform_applicable=sum(r["uniform_applicable"] for r in rows),
                                      empirical_violations=sum(r["empirical_violation"] for r in rows),
                                      oracle_violations=sum(r["oracle_violation"] for r in rows)))
        all_summary[name]=dict(nodes=p,original_edges=len(bn.edges),orders=len(orders),
                               unique_full_orders=len({tuple(o["beta"]) for o in orders}),
                               unique_remaining_orders=len({tuple(o["alpha_R"]) for o in orders}),
                               unique_graphs=len(edge_to_id),evidence_probability=evidence_prob,
                               event_probability=q_target,exact_query_max_abs_error=exact_error,
                               conditional_entropy=entropy,
                               dimension_min=min(r["dimension"] for r in order_rows),
                               dimension_max=max(r["dimension"] for r in order_rows),
                               constructor_ms_median=float(np.median([r["median_ms"] for r in order_rows])),
                               constructor_ms_q25=float(np.quantile([r["median_ms"] for r in order_rows],.25)),
                               constructor_ms_q75=float(np.quantile([r["median_ms"] for r in order_rows],.75)),
                               bounds=bound_summary,selection=selection_summary,
                               elapsed_seconds=time.perf_counter()-t0)
        dump_json(ROOT/"results"/"summary.json",all_summary)
        dump_json(ROOT/"results"/"environment.json",meta)
    print(json.dumps(all_summary,indent=2))


In [3]:
main()


NETWORK child
Distinct conditional graphs: 5
child replicate 1
child replicate 2
child replicate 3
child replicate 4
child replicate 5
child replicate 6
child replicate 7
child replicate 8
child replicate 9
child replicate 10
child replicate 11
child replicate 12
child replicate 13
child replicate 14
child replicate 15
child replicate 16
child replicate 17
child replicate 18
child replicate 19
child replicate 20
NETWORK alarm
Distinct conditional graphs: 50
alarm replicate 1
alarm replicate 2
alarm replicate 3
alarm replicate 4
alarm replicate 5
alarm replicate 6
alarm replicate 7
alarm replicate 8
alarm replicate 9
alarm replicate 10
alarm replicate 11
alarm replicate 12
alarm replicate 13
alarm replicate 14
alarm replicate 15
alarm replicate 16
alarm replicate 17
alarm replicate 18
alarm replicate 19
alarm replicate 20
{
  "child": {
    "nodes": 20,
    "original_edges": 25,
    "orders": 100,
    "unique_full_orders": 100,
    "unique_remaining_orders": 100,
    "unique_graphs": 5,